# Sonar Tracker Evaluation — YOLOv26s + ByteTrack

Runs the YOLOv26s model on all sonar MOT sequences, compares predictions to GT,
and reports standard MOT metrics **separately for each class**.

**Model:** `yolov26s_net_fish_sonar_120e_fair/weights/best.pt`  
**Classes:** fish (class_id = 1) · net (class_id = 2)  
**Metric matching:** IoU ≥ 0.5  

Sonar perspective: top-down, ROV at bottom of image.  
Physical coordinates (metres) are used in spatial error plots.

Run from `tracking/` directory.

In [ ]:
import sys
sys.path.insert(0, "..")

import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from tracking.utils.mot_metrics import (
    load_gt, run_inference, img_dir_for, frame_map,
    collect_images,
    compute_metrics, compute_hota, format_summary, build_summary_table,
    plot_metrics_bar, plot_error_breakdown,
    plot_track_comparison, plot_det_timeline, plot_id_switches,
)

plt.rcParams.update({"figure.dpi": 130, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

# ── Sonar physical mapping ─────────────────────────────────────────────────────
S_W, S_H  = 1200, 700
S_YMAX_M  = 8.0
S_XSPAN_M = S_YMAX_M * np.sin(np.deg2rad(45.0))

def sonar_px_to_m(cx, cy):
    y_m = (S_H - 1 - cy) * (S_YMAX_M / (S_H - 1))
    x_m = cx * (2 * S_XSPAN_M / S_W) - S_XSPAN_M
    return x_m, y_m, np.sqrt(x_m**2 + y_m**2)


In [ ]:
REPO       = Path("..").resolve()
SONAR_ROOT = REPO / "data-processing" / "sonar" / "MOT"
MODEL_PATH = REPO / "runs/detect/outputs/training/net_fish_sonar/yolov26s_net_fish_sonar_120e_fair/weights/best.pt"

# ── Sequence selection ─────────────────────────────────────────────────────────
# List specific sequence names to process, or leave empty to run all.
SELECTED_SEQUENCES = [
    # "2024-08-20_seq1",
    # "2024-08-21_seq2",
]

CONF       = 0.25
IOU_NMS    = 0.45
IOU_MATCH  = 0.5
DEVICE     = "cuda:0"
FRAME_RATE = 30

# ── MOT output ────────────────────────────────────────────────────────────────
MOT_OUTPUT_DIR = Path("outputs/inference_annotation_MOT11")
MOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_seqs = sorted([p.name for p in SONAR_ROOT.iterdir() if p.is_dir()])
SEQUENCES = [s for s in all_seqs if s in SELECTED_SEQUENCES] if SELECTED_SEQUENCES else all_seqs

print(f"Model   : {MODEL_PATH.name}")
print(f"Device  : {DEVICE}")
print(f"Sequences ({len(SEQUENCES)}):")
for s in SEQUENCES:
    print(f"  {s}")


## 1  Run inference on selected sequences
Edit `SELECTED_SEQUENCES` in the cell above to choose which sequences to process.  
Results are cached in memory and saved as MOT .txt files under `outputs/inference_annotation_MOT11/`.  
Re-run this cell to refresh.

In [ ]:
gt_all   = {}  # full GT (all classes)
pred_all = {}  # full predictions
total_frames = {}

for seq in SEQUENCES:
    seq_path = SONAR_ROOT / seq
    idir     = img_dir_for(seq_path, "sonar")

    gt   = load_gt(seq_path / "gt" / "gt.txt", idir)   # all classes
    pred = run_inference(MODEL_PATH, idir,
                         conf=CONF, iou=IOU_NMS,
                         device=DEVICE, frame_rate=FRAME_RATE,
                         desc=seq)

    gt_all[seq]   = gt
    pred_all[seq] = pred
    total_frames[seq] = len(list(idir.glob("*.jpg")))

    # ── Write MOT .txt (same format as tracking-inference_improved_MOT11.py) ──
    mot_lines = [
        f"{int(row.frame_id)},{int(row.track_id)},"
        f"{row.x:.2f},{row.y:.2f},{row.w:.2f},{row.h:.2f},"
        f"1,{int(row.class_id)},1.000000"
        for row in pred.sort_values(["frame_id", "track_id"]).itertuples()
    ] if not pred.empty else []
    mot_path = MOT_OUTPUT_DIR / f"INFERENCE_MOT11_{seq}.txt"
    mot_path.write_text("\n".join(mot_lines), encoding="utf-8")

    for cls, name in [(1, "fish"), (2, "net")]:
        ng  = (gt["class_id"] == cls).sum()
        np_ = (pred["class_id"] == cls).sum() if not pred.empty else 0
        print(f"  {seq}  {name}: GT={ng}  Pred={np_}")
    print(f"  -> {mot_path.name}")


## 2  Compute metrics — fish (class 1)

In [ ]:
gt_fish   = {s: gt_all[s][gt_all[s]["class_id"] == 1].copy() for s in SEQUENCES}
pred_fish = {s: pred_all[s][pred_all[s]["class_id"] == 1].copy()
             if not pred_all[s].empty else pred_all[s] for s in SEQUENCES}

accs_fish    = {}
results_fish = {}

for seq in SEQUENCES:
    acc, summary = compute_metrics(gt_fish[seq], pred_fish[seq], iou_threshold=IOU_MATCH)
    hota         = compute_hota(gt_fish[seq], pred_fish[seq])
    for k, v in hota.items():
        summary[k] = v
    accs_fish[seq]    = acc
    results_fish[seq] = summary
    print(f"{seq[-8:]}  "
          f"HOTA={hota['hota']*100:.1f}%  "
          f"MOTA={float(summary['mota'].iloc[0])*100:.1f}%  "
          f"IDF1={float(summary['idf1'].iloc[0])*100:.1f}%  "
          f"IDSW={int(summary['num_switches'].iloc[0])}")

print("\n── Fish metrics table ──")
display(build_summary_table(results_fish))


## 3  Compute metrics — net (class 2)

In [ ]:
gt_net   = {s: gt_all[s][gt_all[s]["class_id"] == 2].copy() for s in SEQUENCES}
pred_net = {s: pred_all[s][pred_all[s]["class_id"] == 2].copy()
            if not pred_all[s].empty else pred_all[s] for s in SEQUENCES}

accs_net    = {}
results_net = {}

for seq in SEQUENCES:
    if gt_net[seq].empty:
        print(f"{seq[-8:]}: no net GT — skipping")
        continue
    acc, summary = compute_metrics(gt_net[seq], pred_net[seq], iou_threshold=IOU_MATCH)
    hota         = compute_hota(gt_net[seq], pred_net[seq])
    for k, v in hota.items():
        summary[k] = v
    accs_net[seq]    = acc
    results_net[seq] = summary
    print(f"{seq[-8:]}  "
          f"HOTA={hota['hota']*100:.1f}%  "
          f"MOTA={float(summary['mota'].iloc[0])*100:.1f}%  "
          f"IDF1={float(summary['idf1'].iloc[0])*100:.1f}%  "
          f"IDSW={int(summary['num_switches'].iloc[0])}")

if results_net:
    print("\n── Net metrics table ──")
    display(build_summary_table(results_net))


## 4  Metrics bar charts — fish

In [ ]:
fig = plot_metrics_bar(results_fish,
                       metrics=("MOTA", "IDF1", "MOTP", "Recall", "Precision"),
                       title="Sonar — YOLOv26s + ByteTrack  (fish)")
plt.savefig("outputs/sonar_fish_metrics_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 5  Metrics bar charts — net

In [ ]:
if results_net:
    fig = plot_metrics_bar(results_net,
                           metrics=("MOTA", "IDF1", "MOTP", "Recall", "Precision"),
                           title="Sonar — YOLOv26s + ByteTrack  (net)")
    plt.savefig("outputs/sonar_net_metrics_bar.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No net GT found in any sequence.")

## 6  Error breakdown — fish

In [ ]:
fig = plot_error_breakdown(results_fish, title="Sonar — fish error breakdown")
plt.savefig("outputs/sonar_fish_error_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

## 7  GT vs predicted track counts

In [ ]:
fig = plot_track_comparison(gt_fish, pred_fish)
plt.title("Sonar — GT vs predicted fish tracks", fontweight="bold")
plt.savefig("outputs/sonar_track_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8  Detection count per frame — GT vs predicted

In [ ]:
fig = plot_det_timeline(gt_fish, pred_fish, total_frames,
                        title="Sonar — GT vs predicted fish detections per frame")
plt.savefig("outputs/sonar_det_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

## 9  Cumulative ID switches — fish

In [ ]:
fig = plot_id_switches(accs_fish, total_frames,
                       title="Sonar — cumulative ID switches (fish)")
plt.savefig("outputs/sonar_id_switches.png", dpi=150, bbox_inches="tight")
plt.show()

## 10  Net range error (sonar physical space)
Compare predicted vs GT net bounding box centre range (metres from sonar).

In [ ]:
def add_range_m(df):
    cx = df["x"] + df["w"] / 2
    cy = df["y"] + df["h"] / 2
    _, _, rng = sonar_px_to_m(cx, cy)
    df = df.copy()
    df["range_m"] = rng
    return df

fig, axes = plt.subplots(len(SEQUENCES), 1, figsize=(13, 3.5 * len(SEQUENCES)))
if len(SEQUENCES) == 1:
    axes = [axes]

for ax, seq in zip(axes, SEQUENCES):
    gn = add_range_m(gt_net.get(seq, pd.DataFrame()))
    pn = add_range_m(pred_net.get(seq, pd.DataFrame()))

    if gn.empty:
        ax.set_title(f"{seq[-8:]} — no net GT", fontsize=9)
        continue

    gt_r  = gn.groupby("frame_norm")["range_m"].median()
    ax.plot(gt_r.index,  gt_r.values,  lw=1.4, color="#2E7D32", label="GT net range")

    if not pn.empty:
        pr_r = pn.groupby("frame_norm")["range_m"].median()
        ax.plot(pr_r.index, pr_r.values, lw=1.4, color="#1565C0",
                ls="--", label="Pred net range")

        # Error per frame (where both exist)
        merged = gt_r.rename("gt").to_frame().join(pr_r.rename("pr"), how="inner")
        err = (merged["pr"] - merged["gt"]).abs()
        ax2 = ax.twinx()
        ax2.fill_between(err.index, err.values, alpha=0.15, color="#FF9800")
        ax2.plot(err.index, err.values, lw=0.8, color="#FF9800", label=f"abs error (mean={err.mean():.2f} m)")
        ax2.set_ylabel("|error| (m)", color="#FF9800", fontsize=8)
        ax2.tick_params(axis="y", colors="#FF9800")
        ax2.spines[["top", "bottom", "left"]].set_visible(False)
        h2, l2 = ax2.get_legend_handles_labels()
    else:
        h2, l2 = [], []

    ax.set_ylabel("Range (m)")
    ax.set_title(seq[-8:], fontsize=9, loc="left")
    ax.set_ylim(bottom=0)
    h1, l1 = ax.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper right")

axes[-1].set_xlabel("Frame number")
fig.suptitle("Sonar — net range: GT vs predicted  (orange = absolute error)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("outputs/sonar_net_range_error.png", dpi=150, bbox_inches="tight")
plt.show()

## 11  MT / PT / ML breakdown — fish

In [ ]:
seq_names = list(results_fish.keys())
mt_vals = [int(results_fish[s]["mostly_tracked"].iloc[0])   for s in seq_names]
pt_vals = [int(results_fish[s]["partially_tracked"].iloc[0]) for s in seq_names]
ml_vals = [int(results_fish[s]["mostly_lost"].iloc[0])       for s in seq_names]

x = np.arange(len(seq_names))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x, mt_vals, label="MT (>80%)",  color="#2E7D32", edgecolor="white")
ax.bar(x, pt_vals, bottom=mt_vals, label="PT (20–80%)", color="#FFA726", edgecolor="white")
ax.bar(x, ml_vals, bottom=[m+p for m,p in zip(mt_vals,pt_vals)],
       label="ML (<20%)", color="#EF5350", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([s[-8:] for s in seq_names], rotation=12, ha="right", fontsize=8)
ax.set_ylabel("Number of GT tracks")
ax.set_title("Sonar — Mostly / Partially / Mostly-Lost fish tracks", fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/sonar_mt_pt_ml.png", dpi=150, bbox_inches="tight")
plt.show()

## V2  GT vs inference frame comparison
Green = ground truth boxes, red = predicted boxes.  
Set `COMPARE_CLASS` to 1 (fish) or 2 (net).

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
COMPARE_SEQ    = SEQUENCES[0]
COMPARE_FRAMES = [50, 100, 150, 200]   # frame_norm indices (1-indexed sequential)
COMPARE_CLASS  = 1                     # 1 = fish, 2 = net
NCOLS          = 2

GT_COLOR   = "#00CC44"   # green
PRED_COLOR = "#FF4444"   # red

# ── Plot ──────────────────────────────────────────────────────────────────────
idir_c      = img_dir_for(SONAR_ROOT / COMPARE_SEQ, "sonar")
fn_to_img_c = {i + 1: p for i, p in enumerate(collect_images(idir_c))}
gt_c        = gt_all[COMPARE_SEQ][gt_all[COMPARE_SEQ]["class_id"] == COMPARE_CLASS]
pred_c      = pred_all[COMPARE_SEQ][pred_all[COMPARE_SEQ]["class_id"] == COMPARE_CLASS]

valid_frames = [fn for fn in COMPARE_FRAMES if fn in fn_to_img_c]
nrows = (len(valid_frames) + NCOLS - 1) // NCOLS
fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 8, nrows * 5))
axes = np.array(axes).flatten()

for i, fn in enumerate(valid_frames):
    ax = axes[i]
    img_rgb = cv2.cvtColor(cv2.imread(str(fn_to_img_c[fn])), cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    for _, row in gt_c[gt_c["frame_norm"] == fn].iterrows():
        ax.add_patch(plt.Rectangle((row.x, row.y), row.w, row.h,
                                   linewidth=2, edgecolor=GT_COLOR, facecolor="none"))

    for _, row in pred_c[pred_c["frame_norm"] == fn].iterrows():
        ax.add_patch(plt.Rectangle((row.x, row.y), row.w, row.h,
                                   linewidth=2, edgecolor=PRED_COLOR, facecolor="none"))

    n_gt   = (gt_c["frame_norm"]   == fn).sum()
    n_pred = (pred_c["frame_norm"] == fn).sum()
    ax.set_title(f"Frame {fn}  |  GT={n_gt}  Pred={n_pred}", fontsize=9)
    ax.axis("off")

for j in range(len(valid_frames), len(axes)):
    axes[j].set_visible(False)

class_label = {1: "fish", 2: "net"}.get(COMPARE_CLASS, str(COMPARE_CLASS))
fig.legend(
    handles=[
        mpatches.Patch(color=GT_COLOR,   label="Ground truth"),
        mpatches.Patch(color=PRED_COLOR, label="Inference"),
    ],
    loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.04),
)
fig.suptitle(f"{COMPARE_SEQ} — GT vs Inference ({class_label})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"outputs/sonar_gt_vs_pred_{class_label}.png", dpi=150, bbox_inches="tight")
plt.show()
